In [0]:
# Setup dynamic paths - works for any user
import os

# Get current username dynamically
username = spark.sql("SELECT current_user()").collect()[0][0]

# Or use this alternative method:
# username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()

# Build paths dynamically
PROJECT_ROOT = f"/Workspace/Users/{username}/agentic_quality_check"
SRC_PATH = f"{PROJECT_ROOT}/src"

# Add to Python path
import sys
if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

print(f"✅ Configured for user: {username}")
print(f"📁 Project root: {PROJECT_ROOT}")
print(f"📦 Source path: {SRC_PATH}")

In [0]:
%pip install -r /Workspace/Users/gb.burcea@gmail.com/agentic_quality_check/requirements.txt

In [0]:
# Import directly from utils.pdf_parser to bypass table_extractor dependency
from utils.pdf_parser import parse_pdf, extract_headlines
# Test 
pdf_path = "/Volumes/my_catalog/agentic_quality_check_dev/pdfs_volume/Multiplication_check.pdf"
result = parse_pdf(pdf_path)

print(f"Document: {result['filename']}")
print(f"Pages: {result['metadata']['total_pages']}")
print(f"Headlines found: {len(result['headlines'])}\n")

for h in result['headlines']:
    print(f" - [H{h['level']}] Page {h['page']}: {h['text']}")
    print(f" Paragraphs: {len(h['paragraphs'])}")
    print(f" Text: {' '.join(h['paragraphs'])}")

In [0]:
# Final test: Map headlines with correct keywords
print("\n" + "="*70)
print("FINAL TEST: HEADLINE TO DATASET MAPPING (CORRECTED)")
print("="*70)

# Define headline-to-keyword mapping
headline_mappings = [
    {
        'headline': 'Attainment by gender',
        'pdf_term': 'gender',
        'csv_column': 'sex',  # PDF says "gender" but CSV uses "sex"
        'reason': 'PDF uses "gender" terminology but CSV column is named "sex"'
    },
    {
        'headline': 'Attainment by disadvantage status',
        'pdf_term': 'disadvantage',
        'csv_column': 'disadvantage_status',
        'reason': 'Direct match'
    }
]

# Test each mapping
for mapping in headline_mappings:
    print(f"\n{'='*70}")
    print(f"📌 PDF Headline: '{mapping['headline']}'")
    print(f"{'='*70}")
    
    # Find the column in CSV
    matched_col = next((col for col in csv_data['columns'] 
                        if col['name'] == mapping['csv_column']), None)
    
    if matched_col:
        print(f"\n✅ MATCHED to CSV column: {matched_col['name']}")
        print(f"   📊 Type: {matched_col['type']}")
        print(f"   🎯 Role: {matched_col['role']}")
        print(f"   📊 Values: {matched_col['sample_values']}")
        print(f"\n   💡 Mapping note: {mapping['reason']}")
        
        # Show example data filtering
        print(f"\n   📝 Example queries you can run:")
        if matched_col['name'] == 'sex':
            print(f"      • Filter for Boys: WHERE {matched_col['name']} = 'Boys'")
            print(f"      • Filter for Girls: WHERE {matched_col['name']} = 'Girls'")
        elif matched_col['name'] == 'disadvantage_status':
            print(f"      • Filter for Disadvantaged: WHERE {matched_col['name']} = 'Disadvantaged'")
            print(f"      • Filter for Not Disadvantaged: WHERE {matched_col['name']} = 'Not known to be disadvantaged'")
    else:
        print(f"\n❌ Column '{mapping['csv_column']}' not found in CSV")

print("\n" + "="*70)
print("✅ MAPPING TEST COMPLETE - READY FOR EXTRACTION")
print("="*70)
print("\n🚀 Next step: Use these mappings to extract pivot tables from the PDF")
print("   and match them with the corresponding CSV data.")

In [0]:
# Show all CSV columns to find the gender/sex column
print("\n" + "="*70)
print("ALL CSV COLUMNS")
print("="*70 + "\n")

for i, col in enumerate(csv_data['columns'], 1):
    print(f"{i:2d}. {col['name']:40s} | Type: {col['type']:8s} | Role: {col['role']}")
    if col['sample_values']:
        print(f"    Sample: {col['sample_values'][:3]}")

In [0]:
# Test: Map specific headlines to CSV columns (with synonym handling)
from utils.csv_handler import get_csv_metadata
import re

# Define terminology mapping (synonyms/aliases)
TERMINOLOGY_MAP = {
    'gender': ['sex', 'gender'],
    'disadvantage': ['disadvantage', 'disadvantaged']
}

# Pick specific headlines to test
test_headlines = [
    "Attainment by gender",
    "Attainment by disadvantage status"
]

print("="*70)
print("TEST: HEADLINE TO DATASET MAPPING (WITH SYNONYM HANDLING)")
print("="*70)

# Load CSV metadata
csv_path = "/Volumes/my_catalog/agentic_quality_check_dev/csvs_volume/mtc_national_pupil_characteristics_2022_to_2025.csv"
csv_data = get_csv_metadata(csv_path)

print(f"\n📊 CSV: {csv_data['filename']}")
print(f"   Rows: {csv_data['row_count']:,}")
print(f"   Columns: {csv_data['column_count']}\n")

# For each test headline, find matching columns
for headline in test_headlines:
    print(f"\n{'='*70}")
    print(f"📌 Headline: '{headline}'")
    print(f"{'='*70}")
    
    # Extract keywords from headline (filter out common words)
    keywords = [w.lower() for w in re.findall(r'\w+', headline) 
                if len(w) > 3 and w.lower() not in ['attainment', 'status']]
    print(f"\n🔍 Original keywords: {keywords}")
    
    # Expand keywords with synonyms
    expanded_keywords = []
    for keyword in keywords:
        if keyword in TERMINOLOGY_MAP:
            expanded_keywords.extend(TERMINOLOGY_MAP[keyword])
            print(f"   ↳ '{keyword}' expanded to: {TERMINOLOGY_MAP[keyword]}")
        else:
            expanded_keywords.append(keyword)
    
    print(f"\n🔍 Expanded keywords: {list(set(expanded_keywords))}\n")
    
    # Find matching columns
    matched_columns = []
    for col in csv_data['columns']:
        col_name = col['name'].lower()
        # Check if any expanded keyword appears in column name
        matched_keywords = [kw for kw in expanded_keywords if kw in col_name]
        if matched_keywords:
            matched_columns.append({
                'name': col['name'],
                'type': col['type'],
                'role': col['role'],
                'samples': col['sample_values'][:3],
                'matched_via': matched_keywords
            })
    
    if matched_columns:
        print(f"✅ Found {len(matched_columns)} matching column(s):\n")
        for i, col in enumerate(matched_columns, 1):
            print(f"   {i}. {col['name']}")
            print(f"      Matched via: {col['matched_via']}")
            print(f"      Type: {col['type']}")
            print(f"      Role: {col['role']}")
            print(f"      Sample values: {col['samples']}\n")
    else:
        print("❌ No column matches found")
        print("   💡 Suggestion: Add more synonyms to TERMINOLOGY_MAP\n")

print("\n" + "="*70)
print("✅ MAPPING TEST COMPLETE")
print("="*70)

In [0]:
# Extract pivot tables from PDF using parse_pdf with extract_tables=True
from utils.pdf_parser import parse_pdf

print("="*70)
print("STEP 2: EXTRACT PIVOT TABLES FROM PDF")
print("="*70)

pdf_path = "/Volumes/my_catalog/agentic_quality_check_dev/pdfs_volume/Multiplication_check.pdf"

# Parse PDF with table extraction enabled
result = parse_pdf(pdf_path, extract_tables=True)

if 'tables' in result and result['tables']:
    print(f"\n📊 Total tables found: {len(result['tables'])}\n")
    
    # Show all tables to understand the structure
    for idx, table in enumerate(result['tables'], 1):
            print(f"\n{'='*70}")
            print(f"📋 Table {idx} on Page {table['page']}")
            print(f"{'='*70}")
            print(f"   Dimensions: {len(table['data'])} rows × {len(table['data'][0]) if table['data'] else 0} columns")
            
            if table['data']:
                # First row is usually the header
                print(f"   Headers: {table['data'][0]}")
                print(f"\n   First 3 data rows:")
                for i, row in enumerate(table['data'][1:4], 1):
                    print(f"      {i}. {row}")
            print()
else:
    print("\n⚠️  No tables found in PDF")
    print("   Note: parse_pdf with extract_tables=True should return a 'tables' key")

In [0]:
# Test the FULL automated pipeline: PDF extraction → Headline mapping → MOCK table generation
# Using MOCK extractor to avoid slow LLM inference on CPU
import pandas as pd

# Mock the extractor with hardcoded test responses (FAST - no LLM wait)
class MockTableExtractor:
    def extract_table(self, headline, csv_path, column_metadata):
        # Pre-written pandas code for each test case
        if 'gender' in headline['text'].lower():
            pandas_code = """filtered = df[df['sex'].isin(['Boys', 'Girls'])]
result = filtered.pivot_table(index='time_period', columns='sex', values='mtc_score_average', aggfunc='first')"""
        elif 'disadvantage' in headline['text'].lower():
            pandas_code = """filtered = df[df['disadvantage_status'].isin(['Disadvantaged', 'Not known to be disadvantaged'])]
result = filtered.pivot_table(index='time_period', columns='disadvantage_status', values='mtc_score_average', aggfunc='first')"""
        else:
            pandas_code = "result = df.head(10)"
        
        # Execute the code
        try:
            df = pd.read_csv(csv_path)
            namespace = {'df': df, 'pd': pd}
            exec(pandas_code, namespace)
            result_df = namespace.get('result', namespace['df'])
            
            return {
                'headline_id': headline.get('id', 'unknown'),
                'headline_text': headline['text'],
                'extracted_table': result_df.to_dict('records'),
                'filters_applied': {'mock': 'test'},
                'columns_selected': result_df.columns.tolist(),
                'pandas_code': pandas_code,
                'row_count': len(result_df),
                'confidence': 1.0
            }
        except Exception as e:
            return {
                'error': str(e),
                'headline_id': headline.get('id'),
                'headline_text': headline['text'],
                'confidence': 0.0,
                'extracted_table': []
            }

print("="*70)
print("STEP 3: AUTOMATED END-TO-END TABLE EXTRACTION TEST")
print("="*70)

# Step 1: Get the headlines extracted from PDF (from Cell 3)
print("\nStep 1: Using headlines extracted from PDF...")
print(f"   Available headlines: {len(result['headlines'])}")

# Step 2: Filter for the headlines we want to test
test_headline_texts = [
    "Attainment by gender",

]

print(f"\nStep 2: Filtering for test headlines...")
selected_headlines = []
for h in result['headlines']:
    for target_text in test_headline_texts:
        if target_text.lower() in h['text'].lower():
            selected_headlines.append(h)
            print(f"Found: '{h['text']}' (Page {h['page']})")
            break

if not selected_headlines:
    print("\n No matching headlines found!")
    print("   Available headlines:")
    for h in result['headlines'][:10]:
        print(f"      - {h['text']}")
else:
    # Step 3: Initialize the MOCK extractor (instant, no LLM)
    print("\n⚡ Step 3: Using MOCK extractor (instant - no LLM wait!)...")
    print("   Testing pipeline logic without slow CPU inference\n")
    extractor = MockTableExtractor()
    
    csv_path = "/Volumes/my_catalog/agentic_quality_check_dev/csvs_volume/mtc_national_pupil_characteristics_2022_to_2025.csv"
    
    # Step 4: Process each headline automatically
    print("\n⚡ Step 4: Testing extraction with pre-written pandas code...\n")
    
    for idx, headline in enumerate(selected_headlines, 1):
        print(f"\n{'='*70}")
        print(f"TEST {idx}: {headline['text']}")
        print(f"{'='*70}")
        print(f"Page: {headline['page']}")
        print(f"Paragraphs: {len(headline['paragraphs'])}")
        if headline['paragraphs']:
            print(f"📄 Context: {headline['paragraphs'][0][:100]}...")
        print()
        
        try:
            print("⚡ Using pre-written pandas code (no LLM)...")
            
            # Call extract_table with the ACTUAL extracted headline
            extraction_result = extractor.extract_table(
                headline=headline,
                csv_path=csv_path,
                column_metadata=csv_data['columns']  # From Cell 6
            )
            
            print("✅ Extraction complete!\n")
            
            # Check if extraction succeeded or failed
            if 'error' in extraction_result:
                print("❌ EXTRACTION FAILED")
                print(f"\n🔍 Error details: {extraction_result['error']}\n")
                print(f"📊 Confidence: {extraction_result['confidence']}")
                print(f"🆔 Headline ID: {extraction_result['headline_id']}")
                print(f"📝 Headline text: {extraction_result['headline_text']}")
            else:
                print("✅ EXTRACTION SUCCEEDED\n")
                print(f"📊 Extracted {extraction_result['row_count']} rows")
                print(f"📋 Columns: {extraction_result['columns_selected']}")
                print(f"🎯 Filters: {extraction_result['filters_applied']}")
                print(f"💯 Confidence: {extraction_result['confidence']:.2f}\n")
                
                print("Generated Pandas Code:")
                print("-" * 70)
                print(extraction_result['pandas_code'])
                print("-" * 70)
                
                # Convert back to DataFrame for better display
                result_df = pd.DataFrame(extraction_result['extracted_table'])
                print("\n📊 FULL EXTRACTED TABLE:")
                print()
                display(result_df)
                
        except Exception as e:
            print(f"❌ Error: {str(e)}")
            import traceback
            print("\nFull traceback:")
            print(traceback.format_exc())

print("\n" + "="*70)
print("✅ AUTOMATED TABLE EXTRACTION TEST COMPLETE")
print("="*70)

In [0]:
# Test the FULL automated pipeline: PDF extraction → Headline mapping → LLM table generation
# Force reload the module to get latest changes
import sys
if 'utils.table_extractor' in sys.modules:
    import importlib
    importlib.reload(sys.modules['utils.table_extractor'])

from utils.table_extractor import TableExtractor

print("="*70)
print("STEP 3: AUTOMATED END-TO-END TABLE EXTRACTION TEST")
print("="*70)

# Step 1: Get the headlines extracted from PDF (from Cell 3)
print("\nStep 1: Using headlines extracted from PDF...")
print(f"   Available headlines: {len(result['headlines'])}")

# Step 2: Filter for the headlines we want to test
test_headline_texts = [
    "Attainment by gender"
]

print(f"\nStep 2: Filtering for test headlines...")
selected_headlines = []
for h in result['headlines']:
    for target_text in test_headline_texts:
        if target_text.lower() in h['text'].lower():
            selected_headlines.append(h)
            print(f"Found: '{h['text']}' (Page {h['page']})")
            break

if not selected_headlines:
    print("\n No matching headlines found!")
    print("   Available headlines:")
    for h in result['headlines'][:10]:
        print(f"      - {h['text']}")
else:
    # Step 3: Initialize the TableExtractor
    print("\n Step 3: Initializing TableExtractor (loading Phi-3 model)...")
    print("   (This may take 1-2 minutes the first time)\n")
    extractor = TableExtractor()
    
    csv_path = "/Volumes/my_catalog/agentic_quality_check_dev/csvs_volume/mtc_national_pupil_characteristics_2022_to_2025.csv"
    
    # Step 4: Process each headline automatically
    print("\n🤖 Step 4: Passing headlines to LLM for table extraction...\n")
    
    for idx, headline in enumerate(selected_headlines, 1):
        print(f"\n{'='*70}")
        print(f"TEST {idx}: {headline['text']}")
        print(f"{'='*70}")
        print(f"Page: {headline['page']}")
        print(f"Paragraphs: {len(headline['paragraphs'])}")
        if headline['paragraphs']:
            print(f"📄 Context: {headline['paragraphs'][0][:100]}...")
        print()
        
        try:
            print(" sking LLM to analyze headline and generate pivot table code...")
            
            # Call extract_table with the ACTUAL extracted headline
            extraction_result = extractor.extract_table(
                headline=headline,
                csv_path=csv_path,
                column_metadata=csv_data['columns']  # From Cell 6
            )
            
            print("✅ Extraction complete!\n")
            
            # Check if extraction succeeded or failed
            if 'error' in extraction_result:
                print("❌ EXTRACTION FAILED\n")
                print(f"🔍 Error: {extraction_result['error']}\n")
                print(f"📊 Confidence: {extraction_result['confidence']}")
                
                # Show the bad code if available
                if 'pandas_code' in extraction_result:
                    print("\n🐛 LLM Generated Code (with syntax error):")
                    print("-" * 70)
                    print(extraction_result['pandas_code'])
                    print("-" * 70)
            else:
                print("✅ EXTRACTION SUCCEEDED\n")
                print(f"📊 Extracted {extraction_result['row_count']} rows")
                print(f"📋 Columns: {extraction_result['columns_selected']}")
                print(f"🎯 Filters: {extraction_result['filters_applied']}")
                print(f"💯 Confidence: {extraction_result['confidence']:.2f}\n")
                
                print("Generated Pandas Code:")
                print("-" * 70)
                print(extraction_result['pandas_code'])
                print("-" * 70)
                
                print("\n📊 Extracted Table (first 5 rows):")
                for i, row in enumerate(extraction_result['extracted_table'][:5], 1):
                    print(f"   {i}. {row}")
                
        except Exception as e:
            print(f"❌ Error: {str(e)}")
            import traceback
            print("\nFull traceback:")
            print(traceback.format_exc())

print("\n" + "="*70)
print("✅ AUTOMATED TABLE EXTRACTION TEST COMPLETE")
print("="*70)

In [0]:
# Test the FULL automated pipeline: PDF extraction → Headline mapping → LLM table generation
# Force reload the module to get latest changes
import sys
if 'utils.table_extractor' in sys.modules:
    import importlib
    importlib.reload(sys.modules['utils.table_extractor'])

from utils.table_extractor import TableExtractor

print("="*70)
print("STEP 3: AUTOMATED END-TO-END TABLE EXTRACTION TEST")
print("="*70)

# Step 1: Get the headlines extracted from PDF (from Cell 3)
print("\nStep 1: Using headlines extracted from PDF...")
print(f"   Available headlines: {len(result['headlines'])}")

# Step 2: Filter for the headlines we want to test
test_headline_texts = [
    "Attainment by gender"
]

print(f"\nStep 2: Filtering for test headlines...")
selected_headlines = []
for h in result['headlines']:
    for target_text in test_headline_texts:
        if target_text.lower() in h['text'].lower():
            selected_headlines.append(h)
            print(f"Found: '{h['text']}' (Page {h['page']})")
            break

if not selected_headlines:
    print("\n No matching headlines found!")
    print("   Available headlines:")
    for h in result['headlines'][:10]:
        print(f"      - {h['text']}")
else:
    # Step 3: Initialize the TableExtractor
    print("\n Step 3: Initializing TableExtractor (loading Phi-3 model)...")
    print("   (This may take 1-2 minutes the first time)\n")
    extractor = TableExtractor()
    
    csv_path = "/Volumes/my_catalog/agentic_quality_check_dev/csvs_volume/mtc_national_pupil_characteristics_2022_to_2025.csv"
    
    # Step 4: Process each headline automatically
    print("\n🤖 Step 4: Passing headlines to LLM for table extraction...\n")
    
    for idx, headline in enumerate(selected_headlines, 1):
        print(f"\n{'='*70}")
        print(f"TEST {idx}: {headline['text']}")
        print(f"{'='*70}")
        print(f"Page: {headline['page']}")
        print(f"Paragraphs: {len(headline['paragraphs'])}")
        if headline['paragraphs']:
            print(f"📄 Context: {headline['paragraphs'][0][:100]}...")
        print()
        
        try:
            print(" sking LLM to analyze headline and generate pivot table code...")
            
            # Call extract_table with the ACTUAL extracted headline
            extraction_result = extractor.extract_table(
                headline=headline,
                csv_path=csv_path,
                column_metadata=csv_data['columns']  # From Cell 6
            )
            
            print("✅ Extraction complete!\n")
            
            # Check if extraction succeeded or failed
            if 'error' in extraction_result:
                print("❌ EXTRACTION FAILED\n")
                print(f"🔍 Error: {extraction_result['error']}\n")
                print(f"📊 Confidence: {extraction_result['confidence']}")
                
                # Show the bad code if available
                if 'pandas_code' in extraction_result:
                    print("\n🐛 LLM Generated Code (with syntax error):")
                    print("-" * 70)
                    print(extraction_result['pandas_code'])
                    print("-" * 70)
            else:
                print("✅ EXTRACTION SUCCEEDED\n")
                print(f"📊 Extracted {extraction_result['row_count']} rows")
                print(f"📋 Columns: {extraction_result['columns_selected']}")
                print(f"🎯 Filters: {extraction_result['filters_applied']}")
                print(f"💯 Confidence: {extraction_result['confidence']:.2f}\n")
                
                print("Generated Pandas Code:")
                print("-" * 70)
                print(extraction_result['pandas_code'])
                print("-" * 70)
                
                print("\n📊 Extracted Table (first 5 rows):")
                for i, row in enumerate(extraction_result['extracted_table'][:5], 1):
                    print(f"   {i}. {row}")
                
        except Exception as e:
            print(f"❌ Error: {str(e)}")
            import traceback
            print("\nFull traceback:")
            print(traceback.format_exc())

print("\n" + "="*70)
print("✅ AUTOMATED TABLE EXTRACTION TEST COMPLETE")
print("="*70)

In [0]:
# Test the FULL automated pipeline: PDF extraction → Headline mapping → LLM table generation
# Force reload the module to get latest changes
import sys
if 'utils.table_extractor' in sys.modules:
    import importlib
    importlib.reload(sys.modules['utils.table_extractor'])

from utils.table_extractor import TableExtractor

print("="*70)
print("STEP 3: AUTOMATED END-TO-END TABLE EXTRACTION TEST")
print("="*70)

# Step 1: Get the headlines extracted from PDF (from Cell 3)
print("\nStep 1: Using headlines extracted from PDF...")
print(f"   Available headlines: {len(result['headlines'])}")

# Step 2: Filter for the headlines we want to test
test_headline_texts = [
    "Attainment by gender"
]

print(f"\nStep 2: Filtering for test headlines...")
selected_headlines = []
for h in result['headlines']:
    for target_text in test_headline_texts:
        if target_text.lower() in h['text'].lower():
            selected_headlines.append(h)
            print(f"Found: '{h['text']}' (Page {h['page']})")
            break

if not selected_headlines:
    print("\n No matching headlines found!")
    print("   Available headlines:")
    for h in result['headlines'][:10]:
        print(f"      - {h['text']}")
else:
    # Step 3: Initialize the TableExtractor
    print("\n Step 3: Initializing TableExtractor (loading Phi-3 model)...")
    print("   (This may take 1-2 minutes the first time)\n")
    extractor = TableExtractor()
    
    csv_path = "/Volumes/my_catalog/agentic_quality_check_dev/csvs_volume/mtc_national_pupil_characteristics_2022_to_2025.csv"
    
    # Step 4: Process each headline automatically
    print("\n🤖 Step 4: Passing headlines to LLM for table extraction...\n")
    
    for idx, headline in enumerate(selected_headlines, 1):
        print(f"\n{'='*70}")
        print(f"TEST {idx}: {headline['text']}")
        print(f"{'='*70}")
        print(f"Page: {headline['page']}")
        print(f"Paragraphs: {len(headline['paragraphs'])}")
        if headline['paragraphs']:
            print(f"📄 Context: {headline['paragraphs'][0][:100]}...")
        print()
        
        try:
            print(" sking LLM to analyze headline and generate pivot table code...")
            
            # Call extract_table with the ACTUAL extracted headline
            extraction_result = extractor.extract_table(
                headline=headline,
                csv_path=csv_path,
                column_metadata=csv_data['columns']  # From Cell 6
            )
            
            print("✅ Extraction complete!\n")
            
            # Check if extraction succeeded or failed
            if 'error' in extraction_result:
                print("❌ EXTRACTION FAILED\n")
                print(f"🔍 Error: {extraction_result['error']}\n")
                print(f"📊 Confidence: {extraction_result['confidence']}")
                
                # Show the bad code if available
                if 'pandas_code' in extraction_result:
                    print("\n🐛 LLM Generated Code (with syntax error):")
                    print("-" * 70)
                    print(extraction_result['pandas_code'])
                    print("-" * 70)
            else:
                print("✅ EXTRACTION SUCCEEDED\n")
                print(f"📊 Extracted {extraction_result['row_count']} rows")
                print(f"📋 Columns: {extraction_result['columns_selected']}")
                print(f"🎯 Filters: {extraction_result['filters_applied']}")
                print(f"💯 Confidence: {extraction_result['confidence']:.2f}\n")
                
                print("Generated Pandas Code:")
                print("-" * 70)
                print(extraction_result['pandas_code'])
                print("-" * 70)
                
                print("\n📊 Extracted Table (first 5 rows):")
                for i, row in enumerate(extraction_result['extracted_table'][:5], 1):
                    print(f"   {i}. {row}")
                
        except Exception as e:
            print(f"❌ Error: {str(e)}")
            import traceback
            print("\nFull traceback:")
            print(traceback.format_exc())

print("\n" + "="*70)
print("✅ AUTOMATED TABLE EXTRACTION TEST COMPLETE")
print("="*70)

In [0]:
# Install transformers library for LLM-based table extraction
# Using version 4.45.0 for compatibility with Phi-3 model's generate() method
%pip install transformers==4.45.0 torch accelerate --quiet

In [0]:
# Restart Python to load newly installed packages
dbutils.library.restartPython()